In [1]:
import io
import zipfile
import xml.etree.ElementTree as ET

import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [2]:
URL_FLUX = "https://donnees.roulez-eco.fr/opendata/instantane"

reponse = requests.get(URL_FLUX, timeout=120)
reponse.raise_for_status()

archive = zipfile.ZipFile(io.BytesIO(reponse.content))
print(archive.namelist())
contenu_xml = archive.read(archive.namelist()[0])
racine = ET.fromstring(contenu_xml)
print(len(racine), "stations dans le fichier")

['PrixCarburants_instantane.xml']
9827 stations dans le fichier


In [3]:
stations = []
prix = []

for pdv in racine.findall("pdv"):
    id_station = pdv.get("id")
    stations.append({
        "id": id_station,
        "latitude": pdv.get("latitude"),
        "longitude": pdv.get("longitude"),
        "cp": pdv.get("cp"),
        "type_route": pdv.get("pop"),
        "ville": pdv.findtext("ville"),
    })
    for p in pdv.findall("prix"):
        prix.append({
            "id": id_station,
            "carburant": p.get("nom"),
            "valeur": p.get("valeur"),
            "maj": p.get("maj"),
        })

df_stations = pd.DataFrame(stations)
df_prix = pd.DataFrame(prix)
print("Stations :", df_stations.shape)
print("Prix :", df_prix.shape)
df_stations.head()

Stations : (9827, 6)
Prix : (33158, 4)


,id,latitude,longitude,cp,type_route,ville
0,56580001,4806100,-269000,56580,R,Bréhan
1,89100001,4818763,330400,89100,R,SENS
2,5100009,4493400,673300,05100,R,Montgenèvre
3,6110002,4357200,701600,06110,R,Le Cannet
4,33114001,4462300,-78700,33114,R,LE BARP


In [4]:
df_prix.head()

,id,carburant,valeur,maj
0,56580001,Gazole,2.109,2026-11-12 06:55:20
1,56580001,SP95,2.009,2026-11-12 06:55:20
2,56580001,SP98,2.049,2026-11-12 06:55:21
3,89100001,Gazole,2.099,2026-11-23 14:35:41
4,89100001,E85,0.869,2026-11-10 09:49:38


In [5]:
df_stations["latitude"] = pd.to_numeric(df_stations["latitude"], errors="coerce") / 100_000
df_stations["longitude"] = pd.to_numeric(df_stations["longitude"], errors="coerce") / 100_000
df_stations["departement"] = df_stations["cp"].str[:2]

# On garde la France métropolitaine (Corse comprise) grâce à un rectangle de coordonnées
en_metropole = (
    df_stations["latitude"].between(41, 51.5)
    & df_stations["longitude"].between(-5.5, 10)
)
print("Stations hors métropole ou sans coordonnées :", (~en_metropole).sum())
df_stations = df_stations[en_metropole].copy()
print("Stations conservées :", len(df_stations))

Stations hors métropole ou sans coordonnées : 0
Stations conservées : 9827


In [6]:
df_prix["valeur"] = pd.to_numeric(df_prix["valeur"], errors="coerce")
df_prix["maj"] = pd.to_datetime(df_prix["maj"], errors="coerce")

prix_plausible = df_prix["valeur"].between(0.5, 3.5)
date_reference = df_prix["maj"].max()
prix_recent = df_prix["maj"] >= date_reference - pd.Timedelta(days=7)

print("Prix hors de [0,50 ; 3,50] €/L :", (~prix_plausible).sum())
print("Prix de plus de 7 jours :", (~prix_recent).sum())

df_prix = df_prix[prix_plausible & prix_recent & df_prix["id"].isin(df_stations["id"])].copy()
print("Prix conservés :", len(df_prix))
df_prix.groupby("carburant")["valeur"].describe()

Prix hors de [0,50 ; 3,50] €/L : 0
Prix de plus de 7 jours : 6015
Prix conservés : 27143


,count,mean,std,min,25%,50%,75%,max
carburant,,,,,,,,
E10,6817.0,1.946515,0.074156,1.339,1.899,1.934,1.989,2.919
E85,2601.0,0.896208,0.071849,0.659,0.858,0.885,0.919,2.019
GPLc,892.0,1.087086,0.073599,0.866,1.023,1.088,1.149,1.394
Gazole,8250.0,2.105443,0.088974,1.833,2.031,2.089,2.179,2.530
SP95,2062.0,2.007750,0.070597,1.858,1.966,1.990,2.029,2.380
SP98,6521.0,2.046442,0.082581,1.862,1.990,2.015,2.079,2.529
